In [ ]:
import mlflow
import pandas as pd
from mlflow.models import infer_signature
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from mlflow.models import make_metric
from mlflow.metrics.base import MetricValue

In [ ]:
data = load_diabetes()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Pack test data into one DataFrame (mlflow.evaluate expects this format)
test_data = X_test.copy()
test_data["target"] = y_test

In [ ]:
# Let's set an experiment name
mlflow.set_experiment("diabetes_with_evaluate")

In [ ]:
# Train and log a model
with mlflow.start_run(run_name="only_training") as run:
    
    model = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
    model.fit(X_train, y_train)

    signature = infer_signature(X_train, model.predict(X_train))
    
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        name="rf_model",
        signature=signature,
        input_example=X_train.head(1),
        registered_model_name="DiabetesRandomForestModel"
    )

In [ ]:
with mlflow.start_run(run_name="model_evaluation") as eval_run:
    
    results = mlflow.models.evaluate(
        model=model_info.model_uri,         # can also be "runs:/<run_id>/model"
        data=test_data,
        targets="target",
        model_type="regressor",             # regressor/classifier
        evaluators=["default"],             # default includes regression metrics
    )

    print("Evaluation metrics:")
    print(results.metrics)
    print("Artifacts logged:", results.artifacts.keys())

    print(f"Run ID: {run.info.run_id}")

In [ ]:
# you can also evaluate datasets
eval_df = pd.DataFrame({
    "predictions": model.predict(X_test),
    "target": y_test
})

with mlflow.start_run(run_name="dataset_evaluation") as data_eval_run:
    
    results = mlflow.models.evaluate(
        data=eval_df,
        predictions="predictions",
        targets="target",
        model_type="regressor",
    )

    print("Dataset Evaluation metrics:")
    print(results.metrics)

In [ ]:
# simple diabetes-specific business metric
def diabetes_clinical_metric(predictions, targets):
    """Simple diabetes metric: percentage of predictions within clinically acceptable range."""
    
    # Clinical threshold: predictions within +-25 units are considered acceptable
    clinical_threshold = 25
    within_clinical_range = abs(predictions - targets) <= clinical_threshold
    clinical_accuracy = within_clinical_range.mean()
    
    return MetricValue(aggregate_results={"clinical_accuracy": clinical_accuracy})

business_metric = make_metric(eval_fn=diabetes_clinical_metric, greater_is_better=True, name="diabetes_clinical")

with mlflow.start_run(run_name="custom_metric_evaluation") as custom_eval_run:
    
    results = mlflow.models.evaluate(
        model=model_info.model_uri,
        data=test_data,
        targets="target",
        model_type="regressor",
        extra_metrics=[business_metric],
    )

    print("Custom Metric Evaluation metrics:")
    print(results.metrics)